# 《实用Python编程》教学代码
# 第4章 数据处理与分析-part1

## 4.1 数据清洗

### 示例代码4.1 使用 Pandas 从本地 HTML 文件中读取表格

In [1]:
import os, pandas as pd
from p3lib.ch3 import url_to_filename, PageDownloader #从自定义库p3lib中导入第3章网页抓取相关的函数与类

url = 'http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html' # 目标网址
html_file = os.path.join('html_pages', url_to_filename(url) + ".html") # 本地文件路径
print(html_file)

if os.path.exists(html_file) is False:
    page = PageDownloader()
    page.process(url)
assert(os.path.exists(html_file))

tables = pd.read_html(html_file) # 使用pandas读取HTML中的所有表格
tab = tables[1] # 985高校名单在第2个表格中
colleges = tab.values.flatten() # 将所有值展平为一维数组
new_tab = pd.DataFrame(colleges, columns=['高校名称']) # 创建一个新表
new_tab.to_csv("985名单.txt", index=False, header=False) # 将名单保存为TXT文件

html_pages/aHR0cDovL3d3dy5tb2UuZ292LmNuL3NyY3NpdGUvQTIyL3M3MDY1LzIwMDYxMi90MjAwNjEyMDZfMTI4ODMzLmh0bWw.html


In [2]:
new_tab

,高校名称
0,北京大学
1,中国人民大学
2,清华大学
3,北京航空航天大学
4,北京理工大学
5,中国农业大学
6,北京师范大学
7,中央民族大学
8,南开大学
9,天津大学


### 示例代码4.2 基于正则表达式的字符串查找

In [3]:
import re

pattern = r"\d{4}"
pattern_2 = r"^\d{4}\D"
text_list = """2021年安徽省录取分数线
2023年江西省录取分数线
2022年内蒙古自治区录取分数线
2022年上海市录取分数线
202年内蒙古自治区录取分数线
20229年内蒙古自治区录取分数线""".split()
for text in text_list:
    m = re.search(pattern, text)
    if m: # 等同于 if m is not None:
        print("匹配结果", m.group())
    else:
        print("匹配失败")

匹配结果 2021
匹配结果 2023
匹配结果 2022
匹配结果 2022
匹配失败
匹配结果 2022


### 示例代码4.3 正则表达式中的分组匹配

In [4]:
pattern = r"^(\d{4})\D(.+)(省|市|自治区)"
for text in text_list:
    m = re.search(pattern, text)
    if m: # 等同于 if m is not None:
        print("匹配结果", m.group(1, 2, 3))
    else:
        print("匹配失败")

匹配结果 ('2021', '安徽', '省')
匹配结果 ('2023', '江西', '省')
匹配结果 ('2022', '内蒙古', '自治区')
匹配结果 ('2022', '上海', '市')
匹配失败
匹配失败


### 示例代码4.4 使用BeautifulSoup和正则表达式解析网页标题

数据源：https://github.com/li-xirong/python-book/tree/main/pybook-data/html_pages

In [10]:
from bs4 import BeautifulSoup

pattern = r"^(\d{4})\D(.+)(省|市|自治区)"
html_file = 'pybook-data/html_pages/aHR0cHM6Ly9yZHpzLnJ1Yy5lZHUuY24vaW5xdWlyeS9hZG1pc3Npb24vaW5kZXhjbXMv5YyX5LqsLzIwMjMvbGlzdGNtcw.html'

html_str = open(html_file).read() # 请从本书主页下载相关HTML文件到本地
soup = BeautifulSoup(html_str, 'html.parser')

tag = soup.find('h1', class_='y_tit')
title = tag.get_text().strip()
m = re.search(pattern, title) # pattern在示例代码4.3中定义
year, pro, pro_type = m.group(1, 2, 3)

print(title) # 2023年北京市录取分数线
print(year, pro, pro_type) # 2023 北京 市

2023年北京市录取分数线
2023 北京 市


### 示例代码4.5 利用正则表达式提取所有符合条件的字符串

In [34]:
text = "In a hole in the ground there lived a hobbit."
pattern = r"([^\.\s]+)" # 单词模式串（不含空格和.）
str_list = re.findall(pattern, text) # 匹配成功返回字符串列表，否则返回空列表
print(str_list) 

['In', 'a', 'hole', 'in', 'the', 'ground', 'there', 'lived', 'a', 'hobbit']


## 4.2 中文文本处理与分析

### 示例代码4.6 过滤非中文字符

In [13]:
def is_cn_char(c): # 自定义函数，用于判断输入字符是否为汉字
    code = ord(c)
    return code >= 0x4e00 and code <= 0x9fff

text = "2024年，贵州农产品加工转化率达66%。"
chinese = [] # 创建一个空的列表，用于收集中文字符
for c in text: # 通过for循环遍历字符串text中的每个字符
    if is_cn_char(c): # 若为中文字符，则添加到列表中
        chinese.append(c)
print(chinese) 

['年', '贵', '州', '农', '产', '品', '加', '工', '转', '化', '率', '达']


### 示例代码4.7 使用jieba进行中文分词

In [15]:
import jieba
import jieba.posseg as pseg # 用pseg作为jieba.posseg的别名

text = '特朗普私下承认：共和党要守住参议院多数席位非常困难'
words = jieba.cut(text)
print("/".join(words)) # 特朗普/私下/承认/：/共和党/要/守住/参议院/多数/席位/非常/困难

words = pseg.cut(text) # 分词的同时标记词性，返回单词-词性配对序列
res = []
for (word, tag) in words:
    res.append(f"{word}/{tag}") # 用f-string方式拼接word与tag

print("/".join(res))

特朗普/私下/承认/：/共和党/要/守住/参议院/多数/席位/非常/困难
特朗普/nr/私下/n/承认/v/：/x/共和党/nt/要/v/守住/v/参议院/n/多数/m/席位/n/非常/d/困难/an


### 示例代码4.8 调整jieba词典

In [16]:
text = "乒乓球拍卖完了"
words = pseg.cut(text)
print("/".join([word for (word, tag) in words])) # 乒乓球/拍卖完/了

jieba.add_word("乒乓球拍", freq=1000, tag="n") # 将自定义词汇加入到词典中，词汇权重由freq参数指定
words = pseg.cut(text)
print("/".join([word for (word, tag) in words])) # 乒乓球拍/卖完/了

乒乓球/拍卖/完/了
乒乓球拍/卖完/了


### 示例代码4.9 中文文本的简单信息统计

In [17]:
text_file = os.path.join('pybook-data', 'ch4', '笑傲江湖.txt') # 路径拼接
line_nr = 0   # 记录非空行数
char_nr = 0   # 记录字符数
cnchar_nr = 0 # 记录汉字数

with open(text_file, encoding="utf-8") as fr:
    for line in fr: # 逐行从文件对象中读取文本数据
        line = line.strip()
        if not line: # 跳过空行
            continue
        line_nr = line_nr + 1
        char_nr = char_nr + len(line) # 累计每行的字符数
        
        for c in line:
            if is_cn_char(c): # 示例代码4.1中的自定义函数
                cnchar_nr = cnchar_nr + 1 # 累计每行的汉字数
print(f"该文本共有{line_nr}非空行，包含{char_nr}个字符，{cnchar_nr}个汉字")

该文本共有8604非空行，包含982865个字符，831150个汉字


### 示例代码4.10 利用jieba处理大规模中文文本

In [19]:
from collections import Counter

name2count = Counter() # 计数器，用于统计每个人物被提及的行数
co_occur = Counter()   # 计数器，用于统计两个人物共同被提及的行数
NOT_NAMES = set("言语 武功 高强 高明 明白 冷笑 寻思 尼姑 和尚 武林 武林中 齐声 辟邪 少林寺 华山派 青城派 梅庄 黑木崖".split()) # 非人名集合
characters = "令狐冲 任盈盈 盈盈 任大小姐 岳不群 林平之 东方不败 左冷禅 任我行 仪琳 岳灵珊 向问天 余沧海".split() # 主要人物列表

for name in characters:
    jieba.add_word(name, tag="nr", freq=1000)  # 动态调整jieba词汇，使得主要人物能在分词时被正确标记

with open(text_file, encoding="utf-8") as fr:
    for line in fr: # 逐行从文件对象中读取文本数据
        line = line.strip()
        if not line: # 跳过空行
            continue            
        words = pseg.cut(line) # 分词并标记词性
        persons = set() # 集合，用于临时存放人名        
        for (word, tag) in words: # 从分词结果中提取人名，并暂存到persons集合中
            if ('nr' == tag) and (len(word)>1) and (word not in NOT_NAMES):
                name2count[word] += 1
                persons.add(word)      
        persons = list(persons) # 转成列表，用于统计在当前文本中共同出现的人物次数
        for i in range(len(persons)-1):
            for j in range(i+1, len(persons)):
                # 先将人名按字符顺序排序，再合成多元组，保证多元组的唯一性
                pair = tuple(sorted([persons[i], persons[j]])) 
                co_occur[pair] += 1

print(name2count.most_common(20)) # [('令狐冲', 4884), ('岳不群', 1184), ('盈盈', 1010), ...]
print(f"人物对：{len(co_occur)}") 
threshold, edges = 30, 0
for pair, cofreq in co_occur.items():
    if cofreq >= threshold:
        edges += 1
        print(pair, cofreq)

print(f"共现次数不低于{threshold}的人物对: {edges}") 

[('令狐冲', 4884), ('岳不群', 1184), ('盈盈', 1010), ('林平之', 929), ('岳灵珊', 919), ('仪琳', 714), ('田伯光', 699), ('任我行', 525), ('向问天', 516), ('左冷禅', 482), ('师哥', 434), ('余沧海', 379), ('岳夫人', 376), ('师娘', 337), ('东方不败', 320), ('令狐大哥', 276), ('刘正风', 267), ('黑白子', 251), ('林震南', 247), ('劳德诺', 231)]
人物对：14178
('林平之', '林震南') 56
('林震南', '王夫人') 30
('师哥', '林平之') 41
('余沧海', '林平之') 92
('师哥', '陆大有') 64
('令狐冲', '师太') 57
('令狐冲', '林平之') 134
('令狐冲', '师哥') 87
('令狐冲', '仪琳') 240
('仪琳', '田伯光') 107
('令狐冲', '田伯光') 210
('田伯光', '田兄') 51
('令狐冲', '余沧海') 63
('令狐冲', '劳德诺') 48
('令狐冲', '令狐大哥') 56
('令狐大哥', '仪琳') 116
('令狐大哥', '田伯光') 61
('令狐兄', '令狐冲') 37
('令狐兄', '田伯光') 63
('令狐冲', '田兄') 56
('东方不败', '令狐冲') 85
('余沧海', '木高峰') 54
('木高峰', '林平之') 74
('令狐冲', '岳不群') 297
('仪琳', '曲非烟') 48
('令狐冲', '岳先生') 41
('岳不群', '林平之') 70
('岳不群', '师哥') 30
('劳德诺', '岳不群') 39
('岳不群', '岳灵珊') 96
('岳灵珊', '师哥') 112
('令狐冲', '陆大有') 69
('令狐冲', '岳灵珊') 243
('令狐冲', '林师弟') 63
('岳灵珊', '林平之') 170
('岳灵珊', '林师弟') 32
('岳夫人', '师娘') 44
('岳不群', '岳夫人') 111
('岳不群', '师娘') 50
('令狐冲'

## 4.3 基于NumPy的数值数据操作与计算

### 示例代码4.11 几种常见的创建ndarray对象的方式

In [20]:
import numpy as np # 导入NumPy并以np作为别名来使用

zero_0 = np.array([]) # 一个空数组
zero_1 = np.array([0, 0, 0, 0]) # 列表转数组
zero_2 = np.array((0, 0, 0, 0)) # 元组转数组
zero_3 = np.zeros(5) # 长度为5的一维全0数组
zero_4 = np.zeros((5,3)) # 5行3列的二维全0数组
ones = np.ones(5) # 长度为5的一维全1数组，即array([1., 1., 1., 1., 1.])
seq = np.arange(0, 10, 2 ) # array([0, 2, 4, 6, 8])
identity = np.eye(3) # 3x3的单位矩阵
arr = np.array([[1,2,3], [4,5,6]]) # 2行3列的二维数组

### 示例代码4.12 对比ndarray与list

In [21]:
a = [1, 1, 1]
b = [2, 2, 2]
array_a = np.array(a)
array_b = np.array(b)

print(a + b) # [1, 1, 1, 2, 2, 2]
print(array_a + array_b) # [3 3 3]

np.array([1,2]) + np.array([0,0,3]) 

[1, 1, 1, 2, 2, 2]
[3 3 3]


ValueError: operands could not be broadcast together with shapes (2,) (3,) 

### 示例代码4.13 数组保存与读取

In [22]:
a = np.array([1,2,3,4,5,6]) # 长度为6的一维数组
b = a.reshape(3,2) # 改为一个3行2列的二维数组
fname = 'b.npy'

# 将二维数组b存入一个二进制文件（"b.npy"）
with open(fname, 'wb') as fw:
    np.save(fw, b)
    
# 从二进制文件中导入数组
with open(fname, 'rb') as fr:
    c = np.load(fr)

print(c)
print(np.allclose(b,c)) 

[[1 2]
 [3 4]
 [5 6]]
True


### 示例代码4.14 NumPy 布尔索引

In [23]:
a = np.array(range(10))
new_arr = a[a%2==0]
print(a) # [0 1 2 3 4 5 6 7 8 9]
print(new_arr) # [0 2 4 6 8]

[0 1 2 3 4 5 6 7 8 9]
[0 2 4 6 8]


### 示例代码4.15 二维数组点乘与矩阵乘法

In [24]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[1, 0], [0, 1]]) # 单位矩阵，简称单位阵
C = A * B # 点乘 - 对应元素相乘
print("A*B")
print(C)
D = A @ B # 矩阵相乘 - 因B是单位阵，所以D等于A
print("A@B")
print(D)

A*B
[[1 0]
 [0 4]]
A@B
[[1 2]
 [3 4]]


### 示例代码4.16 使用NumPy求解线性方程组

In [25]:
A = np.array([[2,3],[1,-1]]) # 系数矩阵
b = np.array([8,-1]) # 常数向量
solution = np.linalg.solve(A, b)
print(f"方程解: x={solution[0]:.2f}, y={solution[1]:.2f}") # 输出：方程解: x=1.00, y=2.00
# 验证结果：计算 Ax 是否等于 b
is_correct = np.allclose(A @ solution, b)
print("验证结果是否准确:", is_correct) # 输出：验证结果是否准确: True

方程解: x=1.00, y=2.00
验证结果是否准确: True


### 示例代码 4.17 使用 NumPy 进行最小二乘拟合

In [33]:
# 1. 合成数据: y = 2sin(x) + 1.5cos(x) + 高斯噪声（均值为0，标准差为0.2） 
x = np.linspace(0, 10, 51)
y = 2 * np.sin(x) + 1.5 * np.cos(x) + np.random.normal(0, 0.2, len(x))

# 2. 构造“设计矩阵” A，第一列是 sin(x)，第二列是 cos(x)
A = np.column_stack([np.sin(x), np.cos(x)])

# 3. 求解 A * [a, b].T = y，返回值第一个是系数向量
coeffs, residuals, rank, s = np.linalg.lstsq(A, y, rcond=None)
a, b = coeffs

print(f"拟合出的系数: a={a:.2f}, b={b:.2f}") 

拟合出的系数: a=2.01, b=1.53
